In [2]:
%pip install transformers faiss-cpu sentence-transformers PyPDF2

Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.




[notice] A new release of pip is available: 24.0 -> 24.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
%pip install flask
from flask import Flask, request, jsonify

Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.




[notice] A new release of pip is available: 24.0 -> 24.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import os
import PyPDF2

def extract_text_from_pdfs(folder_path):
    corpus = []
    for file_name in os.listdir(folder_path):
        if file_name.endswith(".pdf"):
            file_path = os.path.join(folder_path, file_name)
            text = ""
            with open(file_path, "rb") as file:
                reader = PyPDF2.PdfReader(file)
                for page in reader.pages:
                    text += page.extract_text()
            corpus.append(text)
    return corpus

# Specify the folder path containing PDF files
pdf_folder_path = "data"

# Extract text from all PDFs in the folder
corpus = extract_text_from_pdfs(pdf_folder_path)


In [5]:
from sentence_transformers import SentenceTransformer
import faiss

# Load a pre-trained sentence transformer model
#retriever_model = SentenceTransformer('all-MiniLM-L6-v2')
retriever_model = SentenceTransformer('all-MPNet-base-v2')

# Encode the corpus
corpus_embeddings = retriever_model.encode(corpus)

# Initialize FAISS index
dimension = corpus_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(corpus_embeddings)


C:\Users\Hp\AppData\Roaming\Python\Python312\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange
C:\Users\Hp\AppData\Roaming\Python\Python312\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [6]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# Load the language model and tokenizer
model_name = "t5-large" # Change to "t5-3b" or "t5-11b" if needed
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [7]:
def retrieve_and_generate(query, top_k=3):
    # Encode the query
    query_embedding = retriever_model.encode([query])
    
    # Retrieve top-k documents
    distances, indices = index.search(query_embedding, top_k)
    
    # Extract top-k documents
    retrieved_docs = [corpus[idx] for idx in indices[0]]
    
    # Concatenate retrieved documents
    context = " ".join(retrieved_docs)
    
    # Generate response using the T5 model
    input_text = f"question: {query} context: {context}"
    inputs = tokenizer.encode(input_text, return_tensors='pt', max_length=512, truncation=True)
    outputs = model.generate(inputs, max_length=150, num_beams=2, early_stopping=True)
    generated_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    return generated_response


In [8]:
@app.route('/query', methods=['POST'])
def handle_query():
    data = request.json
    if 'text' not in data:
        return jsonify({"error": "No text provided"}), 400
    
    query = data['text']
    response = retrieve_and_generate(query)
    return jsonify({"response": response})

if __name__ == '__main__':
    app.run(debug=True,port=5001)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5001
Press CTRL+C to quit
 * Restarting with stat


SystemExit: 1

C:\Users\Hp\AppData\Roaming\Python\Python312\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [9]:
# Example usage
query = "Who wrote boundaries ?"
response = retrieve_and_generate(query)
print(response)

Henry Cloud and John Townsend
